# MTRL (Multi-Turn RL) Dogfooding Notebook

This notebook covers the end-to-end MTRL workflow: **Train → Evaluate → Deploy**.

## Scenarios

1. **Bedrock AgentCore** — Train with AgentCore runtime, evaluate, deploy via ModelBuilder
2. **3P Agent (Lambda)** — Train with Lambda agent, evaluate, deploy via ModelBuilder
3. **Base Model Evaluation** — Evaluate a base JumpStart model (no training)

## Prerequisites

- Valid AWS credentials (`ada credentials update --provider=isengard --account=742774200982 --role=Admin --once`)
- Beta endpoint access for Job APIs
- Bedrock AgentCore runtime deployed (or Lambda agent available)
- Training dataset in S3 (parquet format)


---
## Setup

In [ ]:
import sys, os

# Point to local source (remove these two lines if using pip-installed package)
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'sagemaker-train', 'src')))
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'sagemaker-core', 'src')))
for mod in list(sys.modules.keys()):
    if 'sagemaker' in mod:
        del sys.modules[mod]

# TODO: Remove beta endpoint post-launch
os.environ['SAGEMAKER_ENDPOINT'] = 'https://sagemaker.gamma.us-west-2.ml-platform.aws.a2z.com'
os.environ['SAGEMAKER_REGION'] = 'us-west-2'
os.environ['AWS_REGION'] = 'us-west-2'

# ── Configuration ──────────────────────────────────────────────────────
REGION = 'us-west-2'
ROLE = 'arn:aws:iam::742774200982:role/Admin'
BASE_MODEL = 'openai-reasoning-gpt-oss-20b'

# Agent environments
AGENTCORE_ARN = 'arn:aws:bedrock-agentcore:us-west-2:742774200982:runtime/gsm8k_rft_agent-6OoryL7Uz6'
LAMBDA_AGENT_ARN = 'arn:aws:lambda:us-west-2:742774200982:function:SageMaker-agent-adapter-gsm8k-3p-eval'

# Data & output
DATASET = 's3://sagemaker-rft-beta-742774200982/prompts/gsm8k_small/prompts.parquet'
S3_OUTPUT = 's3://sagemaker-us-west-2-742774200982/model-evaluation/dogfooding/'
MLFLOW_ARN = 'arn:aws:sagemaker:us-west-2:742774200982:mlflow-tracking-server/rft-beta-mlflow'

print('Setup complete.')

---
## Supported Models

Check which models support MTRL training and evaluation before starting.


In [ ]:
from sagemaker.train.multi_turn_rl_trainer import MultiTurnRLTrainer
from sagemaker.train.evaluate import MultiTurnRLEvaluator

# List models supported for MTRL training
training_models = MultiTurnRLEvaluator.list_supported_models()
print(f"Models supported for MTRL training ({len(training_models)}):")
for m in training_models:
    print(f"  - {m}")



---
# Scenario 1: Bedrock AgentCore — Train → Evaluate → Deploy

Uses a Bedrock AgentCore runtime as the agent environment.

## 1.1 Train with AgentCore

In [ ]:
from sagemaker.train.multi_turn_rl_trainer import MultiTurnRLTrainer

trainer_agentcore = MultiTurnRLTrainer(
    model=BASE_MODEL,
    agent_env=AGENTCORE_ARN,
    training_dataset=DATASET,
    mlflow_resource_arn=MLFLOW_ARN,
    s3_output_path=f"{S3_OUTPUT}scenario1/train/",
    role=ROLE,
    accept_eula=True,
)

# Set minimal hyperparameters for fast dogfooding runs
trainer_agentcore.hyperparameters.max_epochs = 1
trainer_agentcore.hyperparameters.global_batch_size = 2
trainer_agentcore.hyperparameters.max_steps = 12

print("Trainer configured with minimal hyperparameters:")
print(trainer_agentcore.hyperparameters.to_dict())


In [ ]:
# Start training and wait for completion
job_agentcore = trainer_agentcore.train(wait=True)

print(f'Job: {job_agentcore.job_name}')
print(f'Status: {job_agentcore.job_status}')
print(f'Output Model Package: {job_agentcore.output_model_package_arn}')

In [ ]:
from pprint import pprint
from sagemaker.core.resources import  Job
job = Job.get(job_name="openai-reasoning-gpt-oss-20b-mtrl-20260512103821")

pprint(job.__dict__)


## 1.2 Evaluate the Fine-Tuned Model

In [ ]:

## Setup
import sys, os

# Point to local source (remove these two lines if using pip-installed package)
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'sagemaker-train', 'src')))
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'sagemaker-core', 'src')))
for mod in list(sys.modules.keys()):
    if 'sagemaker' in mod:
        del sys.modules[mod]

# TODO: Remove beta endpoint post-launch
os.environ['SAGEMAKER_ENDPOINT'] = 'https://sagemaker.gamma.us-west-2.ml-platform.aws.a2z.com'
os.environ['SAGEMAKER_REGION'] = 'us-west-2'
os.environ['AWS_REGION'] = 'us-west-2'

# ── Configuration ──────────────────────────────────────────────────────
REGION = 'us-west-2'
ROLE = 'arn:aws:iam::742774200982:role/Admin'
BASE_MODEL = 'openai-reasoning-gpt-oss-20b'

# Agent environments
AGENTCORE_ARN = 'arn:aws:bedrock-agentcore:us-west-2:742774200982:runtime/gsm8k_rft_agent-6OoryL7Uz6'
LAMBDA_AGENT_ARN = 'arn:aws:lambda:us-west-2:742774200982:function:SageMaker-agent-adapter-gsm8k-3p-eval'

# Data & output
DATASET = 's3://sagemaker-rft-beta-742774200982/prompts/gsm8k_small/prompts.parquet'
S3_OUTPUT = 's3://sagemaker-us-west-2-742774200982/model-evaluation/dogfooding/'
MLFLOW_ARN = 'arn:aws:sagemaker:us-west-2:742774200982:mlflow-tracking-server/rft-beta-mlflow'

print('Setup complete.')

from sagemaker.train.evaluate import MultiTurnRLEvaluator
from sagemaker.train.multi_turn_rl_trainer import MultiTurnRLTrainer


trainer_agentcore = MultiTurnRLTrainer.attach(job_name="openai-reasoning-gpt-oss-20b-mtrl-20260512082005")
# Pass the trainer directly — evaluator auto-resolves the output model package
evaluator_agentcore = MultiTurnRLEvaluator(
    model=trainer_agentcore,
    dataset=DATASET,
    agent_config=AGENTCORE_ARN,
    s3_output_path=f'{S3_OUTPUT}scenario1/eval/',
    mlflow_resource_arn=MLFLOW_ARN,
    role=ROLE,
)

exec_s1 = evaluator_agentcore.evaluate()
print(f'Evaluation started: {exec_s1.arn}')

In [ ]:
# Wait for evaluation to complete
exec_s1.wait()
print(f'Evaluation status: {exec_s1.status.overall_status}')

## 1.3 Deploy with ModelBuilder

In [ ]:
import uuid
import sys, os

# Point to local source (remove these two lines if using pip-installed package)
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'sagemaker-serve', 'src')))
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'sagemaker-train', 'src')))
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'sagemaker-core', 'src')))
for mod in list(sys.modules.keys()):
    if 'sagemaker' in mod:
        del sys.modules[mod]

# TODO: Remove beta endpoint post-launch
os.environ['SAGEMAKER_ENDPOINT'] = 'https://sagemaker.gamma.us-west-2.ml-platform.aws.a2z.com'
os.environ['SAGEMAKER_REGION'] = 'us-west-2'
os.environ['AWS_REGION'] = 'us-west-2'

# ── Configuration ──────────────────────────────────────────────────────
REGION = 'us-west-2'
ROLE = 'arn:aws:iam::742774200982:role/Admin'
BASE_MODEL = 'openai-reasoning-gpt-oss-20b'

# Agent environments
AGENTCORE_ARN = 'arn:aws:bedrock-agentcore:us-west-2:742774200982:runtime/gsm8k_rft_agent-6OoryL7Uz6'
LAMBDA_AGENT_ARN = 'arn:aws:lambda:us-west-2:742774200982:function:SageMaker-agent-adapter-gsm8k-3p-eval'

# Data & output
DATASET = 's3://sagemaker-rft-beta-742774200982/prompts/gsm8k_small/prompts.parquet'
S3_OUTPUT = 's3://sagemaker-us-west-2-742774200982/model-evaluation/dogfooding/'
MLFLOW_ARN = 'arn:aws:sagemaker:us-west-2:742774200982:mlflow-tracking-server/rft-beta-mlflow'

print('Setup complete.')

from sagemaker.serve import ModelBuilder
from sagemaker.core.resources import ModelPackage

# Get the output model package from the completed training job
model_package = ModelPackage.get(
    # model_package_name=job_agentcore.output_model_package_arn
   #  model_package_name="arn:aws:sagemaker:us-west-2:742774200982:model-package/openai-reasoning-gpt-oss-20b-mtrl-mpg/9"
    model_package_name="arn:aws:sagemaker:us-west-2:742774200982:model-package/openai-reasoning-gpt-oss-20b-mtrl-mpg/24"
)
# Build and deploy
model_builder = ModelBuilder(
                             model=model_package,
                             image_uri="763104351884.dkr.ecr.us-west-2.amazonaws.com/djl-inference:0.36.0-lmi24.0.0-cu129",
                             instance_type="ml.g6e.48xlarge"
                             )
model_builder.accept_eula = True
model_builder.build()

endpoint = model_builder.deploy(
    endpoint_name=f"endpoint-{uuid.uuid4().hex[:10]}",
    instance_type="ml.g6e.48xlarge",
    initial_instance_count=1,
)

In [ ]:
import json
import sys, os

# # TODO: Remove beta endpoint post-launch
os.environ['SAGEMAKER_ENDPOINT'] = 'https://maeveruntime.loadtest.us-west-2.ml-platform.aws.a2z.com'
# os.environ['SAGEMAKER_REGION'] = 'us-west-2'
# os.environ['AWS_REGION'] = 'us-west-2'
# # Point to local source (remove these two lines if using pip-installed package)
# sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'sagemaker-serve', 'src')))
# sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'sagemaker-train', 'src')))
# sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'sagemaker-core', 'src')))
# for mod in list(sys.modules.keys()):
#     if 'sagemaker' in mod:
#         del sys.modules[mod]



from sagemaker.core.resources import Endpoint

response = endpoint.invoke(
    body=json.dumps({
        "model": "/opt/ml/model",
        "messages": [{"role": "user", "content": "What is 25 * 4?"}],
        "max_tokens": 200,
        "stream": False
    }).encode('utf-8'),
    content_type="application/json"
)


---
# Scenario 2: 3P Agent (Lambda) — Train → Evaluate → Deploy

Uses a Lambda function as the agent environment. This is the pattern for
third-party agent integrations (LangChain, Strands, custom EKS/Fargate agents).

## 2.1 Train with Lambda Agent

In [ ]:
from sagemaker.train.multi_turn_rl_trainer import MultiTurnRLTrainer
from sagemaker.train.agent_lambda import AgentLambda

LAMBDA_AGENT_ARN = 'arn:aws:lambda:us-west-2:742774200982:function:SageMaker-AgentConnector-gamma-1-1777398957761'


# Option A: Use an existing Lambda ARN directly
trainer_lambda = MultiTurnRLTrainer(
    model=BASE_MODEL,
    agent_env=LAMBDA_AGENT_ARN,  # Lambda ARN string
    training_dataset=DATASET,
    mlflow_resource_arn=MLFLOW_ARN,
    s3_output_path=f"{S3_OUTPUT}scenario2/train/",
    role=ROLE,
    accept_eula=True,
)

# Set minimal hyperparameters for fast dogfooding runs
trainer_lambda.hyperparameters.max_epochs = 1
trainer_lambda.hyperparameters.global_batch_size = 2
trainer_lambda.hyperparameters.max_steps = 12

print("Lambda trainer configured with minimal hyperparameters:")
print(trainer_lambda.hyperparameters.to_dict())


In [ ]:
# Start training and wait
job_lambda = trainer_lambda.train(wait=True)

print(f'Job: {job_lambda.job_name}')
print(f'Status: {job_lambda.job_status}')
print(f'Output Model Package: {job_lambda.output_model_package_arn}')

## 2.2 Evaluate the Fine-Tuned Model (3P Agent)

In [ ]:
from sagemaker.train.evaluate import MultiTurnRLEvaluator

# Pass the Lambda trainer directly
evaluator_lambda = MultiTurnRLEvaluator(
    model=trainer_lambda,
    dataset=DATASET,
    s3_output_path=f'{S3_OUTPUT}scenario2/eval/',
    mlflow_resource_arn=MLFLOW_ARN,
    role=ROLE,
)

exec_s2 = evaluator_lambda.evaluate()
print(f'Evaluation started: {exec_s2.arn}')

In [ ]:
# Wait for evaluation to complete
exec_s2.wait()
print(f'Evaluation status: {exec_s2.status.overall_status}')

## 2.3 Deploy with ModelBuilder

In [ ]:
from sagemaker.serve import ModelBuilder
from sagemaker.core.resources import ModelPackage

# Get the output model package from the Lambda training job
model_package_lambda = ModelPackage.get(
    model_package_name=job_lambda.output_model_package_arn
)

# Build and deploy
model_builder = ModelBuilder(model=model_package_lambda)
model_builder.accept_eula = True
model = model_builder.build()

endpoint = model.deploy(
    initial_instance_count=1,
    instance_type='ml.g5.48xlarge',
)

print(f'Endpoint: {endpoint.endpoint_name}')

---
# Scenario 3: Base Model Evaluation (No Training)

Evaluate a base JumpStart model directly without fine-tuning.
Requires `agent_config` since there's no trainer to auto-resolve from.

In [ ]:
from sagemaker.train.evaluate import MultiTurnRLEvaluator

# Evaluate base model with AgentCore runtime
evaluator_base = MultiTurnRLEvaluator(
    model=BASE_MODEL,                    # JumpStart model ID
    dataset=DATASET,
    agent_config=AGENTCORE_ARN,          # Required when model is not a trainer
    s3_output_path=f'{S3_OUTPUT}scenario3/eval-agentcore/',
    mlflow_resource_arn=MLFLOW_ARN,
    role=ROLE,
    accept_eula=True,
)

exec_s3_agentcore = evaluator_base.evaluate()
print(f'Base model eval (AgentCore): {exec_s3_agentcore.arn}')

In [ ]:
# Evaluate base model with Lambda agent
evaluator_base_lambda = MultiTurnRLEvaluator(
    model=BASE_MODEL,
    dataset=DATASET,
    agent_config=LAMBDA_AGENT_ARN,       # Lambda ARN works too
    s3_output_path=f'{S3_OUTPUT}scenario3/eval-lambda/',
    mlflow_resource_arn=MLFLOW_ARN,
    role=ROLE,
    accept_eula=True,
)

exec_s3_lambda = evaluator_base_lambda.evaluate()
print(f'Base model eval (Lambda): {exec_s3_lambda.arn}')

In [ ]:
# Wait for both evaluations
exec_s3_agentcore.wait()
print(f'AgentCore eval: {exec_s3_agentcore.status.overall_status}')

exec_s3_lambda.wait()
print(f'Lambda eval: {exec_s3_lambda.status.overall_status}')

---
# Discovery & Utilities

In [ ]:
# List all models that support MTRL
from sagemaker.train.multi_turn_rl_trainer import MultiTurnRLTrainer

supported = MultiTurnRLTrainer.list_supported_models()
print(f'Supported MTRL models ({len(supported)}):')
for m in supported:
    print(f'  - {m}')

In [ ]:
# List Bedrock AgentCore runtimes
import json
runtimes = MultiTurnRLTrainer.list_bedrock_agentcore_runtimes()
print(f'AgentCore runtimes ({len(runtimes)}):')
for rt in runtimes:
    mtrl = ' [MTRL]' if rt.get('tags', {}).get('sagemaker-mtrl-compatible') else ''
    print(f'  - {rt["name"]} ({rt["status"]}){mtrl} → {rt["arn"]}')

In [ ]:
# List all MTRL evaluation executions
from sagemaker.train.evaluate import MultiTurnRLEvaluator

all_evals = list(MultiTurnRLEvaluator.get_all(region=REGION))
print(f'Total MTRL evaluations: {len(all_evals)}')
for ex in all_evals[:10]:
    print(f'  {ex.name}: {ex.status.overall_status}')

---
# Bedrock Deployment (Alternative to SageMaker Endpoints)

Instead of deploying to a SageMaker endpoint, you can deploy fine-tuned models
to Amazon Bedrock using `BedrockModelBuilder`. This supports both Nova models
(via `create_custom_model`) and non-Nova models (via `create_model_import_job`).


## Bedrock Deploy — From AgentCore Training (Scenario 1)

In [ ]:
from sagemaker.serve.bedrock_model_builder import BedrockModelBuilder
from sagemaker.core.resources import ModelPackage

# Get the output model package from training
model_package = ModelPackage.get(
    model_package_name="arn:aws:sagemaker:us-west-2:742774200982:model-package/openai-reasoning-gpt-oss-20b-mtrl-mpg/9"
)

# Deploy to Bedrock
bedrock_builder = BedrockModelBuilder(model=model_package)

response = bedrock_builder.deploy(
    #custom_model_name="mtrl-agentcore-finetuned",  # For Nova models
    imported_model_name="mtrl-agentcore-finetuned",  # For non-Nova models
    role_arn=ROLE,
    deployment_name="mtrl-agentcore-deployment",
)

print(f"Bedrock deployment response: {response}")


In [ ]:
import json
import boto3

# Get the model ARN from the deployment response
model_arn = response.get("deploymentArn") or response.get("modelArn")

bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)

invoke_response = bedrock_runtime.invoke_model(
    modelId=model_arn,
    body=json.dumps({
        "messages": [{"role": "user", "content": "What is 15 * 7?"}],
        "max_tokens": 256,
    }),
    contentType="application/json",
)

result = json.loads(invoke_response["body"].read())
print(json.dumps(result, indent=2))

## Bedrock Deploy — From Lambda Training (Scenario 2)

In [ ]:
from sagemaker.serve.bedrock_model_builder import BedrockModelBuilder
from sagemaker.core.resources import ModelPackage

# Get the output model package from Lambda training
model_package_lambda = ModelPackage.get(
    model_package_name=job_lambda.output_model_package_arn
)

# Deploy to Bedrock
bedrock_builder = BedrockModelBuilder(model=model_package_lambda)

response = bedrock_builder.deploy(
    custom_model_name="mtrl-lambda-finetuned",
    role_arn=ROLE,
    deployment_name="mtrl-lambda-deployment",
)

print(f"Bedrock deployment response: {response}")


## Invoke Bedrock Deployed Model

In [ ]:
import json
import boto3

# Get the model ARN from the deployment response
model_arn = response.get("deploymentArn") or response.get("modelArn")

bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)

invoke_response = bedrock_runtime.invoke_model(
    modelId=model_arn,
    body=json.dumps({
        "messages": [{"role": "user", "content": "What is 15 * 7?"}],
        "max_tokens": 256,
    }),
    contentType="application/json",
)

result = json.loads(invoke_response["body"].read())
print(json.dumps(result, indent=2))


---
# Cleanup

Delete endpoints created during this notebook to avoid ongoing charges.

In [ ]:
# Uncomment to delete endpoints
# predictor.delete_endpoint()
# print('Endpoint deleted.')